# Raman Spectroscopy — Valid Samples Combined Dataset

Dataset: `combined_raman_spectra.xlsx` — 44 milk Raman spectra (Amul, Heritage, Nandani, Nestle) across varying milk types, temperature/pH/heating conditions, and laser wavelengths.

Applies the **standard processing pipeline** used throughout this project — identical order and intrinsic parameters to `raman_consensus.ipynb`, `raman_qc.ipynb`, `raman_adulterated_dataset.ipynb`, and `Final/raman_evaluation_dataset.ipynb`.

**Pipeline**
1. Load spectra & parse sample metadata from column names
2. Remove spectra with non-standard laser wavelengths (633 nm & 785 nm) — keep 532 nm only
3. Duplicate detection (exact, correlation, L2) — applied **before** preprocessing
4. Preprocessing: Savitzky–Golay smoothing → ALS baseline correction → non-negative clip → L² normalisation
5. Functional data representation: cubic B-spline (order 4), interior knots every 10th point
6. Save the valid-samples combined dataset to Excel

> **Data availability:** The raw LabRAM `.txt` spectra are not included in this repository because the final datasets are provided in `data/`. This notebook reads `combined_raman_spectra.xlsx`, compiled from those spectra; the resulting dataset is provided (aligned) as `data/raman_valid_samples_combined_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.signal import savgol_filter
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from scipy.interpolate import make_lsq_spline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

IN_PATH = 'combined_raman_spectra.xlsx'

## 1. Load Spectra & Parse Sample Metadata

In [ ]:
def parse_column(col):
    """Extract sample metadata from a combined_raman_spectra.xlsx column name."""
    stem = col.replace('.txt', '')
    lower = stem.lower()

    brand = next((b for b in ['Amul', 'Heritage', 'Nandani', 'Nestle'] if b.lower() in lower), 'unknown')

    if 'full cream' in lower:
        milk_type = 'Full Cream'
    elif 'toned' in lower:
        milk_type = 'Toned'
    elif 'special' in lower:
        milk_type = 'Special'
    else:
        milk_type = 'unknown'

    if 'after 24' in lower:
        condition = 'RT after 24h'
    elif 'after heating' in lower:
        condition = 'After Heating'
    elif 'basic' in lower:
        condition = 'Basic (pH)'
    elif 'acidic' in lower:
        condition = 'Acidic (pH)'
    elif 'cold' in lower:
        condition = 'Cold'
    elif re.search(r'\brt\b', lower):
        condition = 'Room Temp (RT)'
    else:
        condition = 'unknown'

    laser  = next(iter(re.findall(r'\d+nm', stem)), 'unknown')
    power  = next(iter(re.findall(r'\d+%', stem)), 'unknown')
    hole   = next(iter(re.findall(r'\d+hole', stem)), 'n/a')
    repeat = 'Yes' if '_Re' in stem else 'No'

    return {
        'Column': col, 'Label': stem, 'Brand': brand, 'Milk Type': milk_type,
        'Condition': condition, 'Laser': laser, 'Power': power, 'Hole': hole, 'Repeat': repeat,
    }

df = pd.read_excel(IN_PATH)
raman_shift = df['Raman Shift'].values.astype(float)
spectrum_cols = [c for c in df.columns if c != 'Raman Shift']

meta_df = pd.DataFrame([parse_column(c) for c in spectrum_cols])

print(f'Loaded {IN_PATH}')
print(f'Spectra (columns)     : {len(spectrum_cols)}')
print(f'Wavenumber points     : {len(raman_shift)}')
print(f'Wavenumber range      : {raman_shift.min():.1f} – {raman_shift.max():.1f} cm⁻¹')
print(f'Missing values        : {df[spectrum_cols].isna().sum().sum()}')
print(f'Unrecognised metadata : {(meta_df[["Brand", "Milk Type", "Condition"]] == "unknown").any(axis=1).sum()} spectra')
print()
print('Brand × Milk Type coverage:')
print(meta_df.pivot_table(index='Brand', columns='Milk Type', values='Column', aggfunc='count', fill_value=0).to_string())

## 2. Remove Non-Standard Laser Spectra

Keep only 532 nm acquisitions — matches `raman_consensus.ipynb`, which excludes the 633 nm and 785 nm Heritage comparison spectra before building the consensus.

In [ ]:
exclude = [c for c in spectrum_cols if '633nm' in c or '785nm' in c]
print(f'Dropping {len(exclude)} non-standard laser spectra:')
for e in exclude:
    print(f'  • {e}')

keep_cols = [c for c in spectrum_cols if c not in exclude]
dataset_raw = df.set_index('Raman Shift')[keep_cols].copy()
dataset_raw.index.name = 'Raman Shift'
meta_df = meta_df[meta_df['Column'].isin(keep_cols)].reset_index(drop=True)

print(f'\nRemaining spectra: {dataset_raw.shape[1]}')

## 3. Duplicate Detection (Prior to Preprocessing)

Three complementary checks, applied to the raw (pre-preprocessing) intensities — identical criteria to `Final/raman_evaluation_dataset.ipynb` and `raman_adulterated_dataset.ipynb`:

| Check | Method | Criterion |
|-------|--------|----------|
| **Exact duplicates** | Column-wise equality | All intensity values identical |
| **Near-duplicates (correlation)** | Pearson r between every pair | r > 0.99999 |
| **Near-duplicates (L2 distance)** | L2 distance between every pair | L2 < 1e-6 |

In [ ]:
cols   = list(dataset_raw.columns)
F      = dataset_raw.values
n_spec = len(cols)

exact_dup_pairs = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        if np.array_equal(F[:, i], F[:, j]):
            exact_dup_pairs.append((cols[i], cols[j]))
print(f'Exact duplicate pairs found: {len(exact_dup_pairs)}')
for a, b in exact_dup_pairs:
    print(f'  {a}  ==  {b}')

CORR_THRESHOLD = 0.99999
corr_matrix = np.corrcoef(F.T)
near_dup_corr = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        r = corr_matrix[i, j]
        if r > CORR_THRESHOLD:
            near_dup_corr.append((cols[i], cols[j], round(r, 8)))
print(f'\nNear-duplicate pairs (r > {CORR_THRESHOLD}): {len(near_dup_corr)}')
for a, b, r in near_dup_corr:
    print(f'  {a}  ↔  {b}   r = {r}')

L2_THRESHOLD = 1e-6
near_dup_l2 = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        d = np.linalg.norm(F[:, i] - F[:, j])
        if d < L2_THRESHOLD:
            near_dup_l2.append((cols[i], cols[j], d))
print(f'\nNear-duplicate pairs (L2 < {L2_THRESHOLD}): {len(near_dup_l2)}')
for a, b, d in near_dup_l2:
    print(f'  {a}  ↔  {b}   L2 = {d:.2e}')

## 4. Remove Duplicates & Build Clean Dataset

In [ ]:
to_drop = set()

for a, b in exact_dup_pairs:
    to_drop.add(b)
    print(f'Exact duplicate — dropping: {b}  (kept: {a})')

for a, b, r in near_dup_corr:
    if b not in to_drop and a not in to_drop:
        to_drop.add(b)
        print(f'Near-duplicate (r={r:.7f}) — dropping: {b}  (kept: {a})')

for a, b, d in near_dup_l2:
    if b not in to_drop and a not in to_drop:
        to_drop.add(b)
        print(f'Near-duplicate (L2={d:.2e}) — dropping: {b}  (kept: {a})')

if not to_drop:
    print('No duplicates detected across all three checks — all spectra are unique.')

dataset_clean = dataset_raw.drop(columns=list(to_drop))
meta_df = meta_df[~meta_df['Column'].isin(to_drop)].reset_index(drop=True)

print(f'\nSpectra before deduplication : {dataset_raw.shape[1]}')
print(f'Duplicates removed           : {len(to_drop)}')
print(f'Spectra in final dataset     : {dataset_clean.shape[1]}')

## 5. Preprocessing Pipeline

Identical order and intrinsic parameters to every other notebook in this project (`raman_eda.ipynb`, `raman_consensus.ipynb`, `raman_qc.ipynb`, `raman_adulterated_dataset.ipynb`, `Final/raman_evaluation_dataset.ipynb`):

1. **Savitzky–Golay smoothing** — window = 5 points, polynomial order = 2
2. **ALS baseline correction** (Eilers & Boelens 2005) — λ = 1×10⁵, p = 0.01, 10 iterations
3. **Non-negative clip** — set any negative residuals to zero
4. **L² normalisation** — scale each spectrum to unit Euclidean norm

In [ ]:
def sg_smooth(y, window=5, polyorder=2):
    return savgol_filter(y, window_length=window, polyorder=polyorder)

def als_baseline(y, lam=1e5, p=0.01, n_iter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, format='csc')
        Z = spsolve(W + H, w * y)
        w = np.where(y > Z, p, 1 - p)
    return Z

def l2_normalize(y):
    n = np.linalg.norm(y)
    return y / n if n > 0 else y

def preprocess(y):
    y = sg_smooth(y)          # 1. Smooth
    y = y - als_baseline(y)   # 2. Subtract ALS baseline
    y = np.clip(y, 0, None)   # 3. Non-negative clip
    return l2_normalize(y)    # 4. L2 normalise

preprocessed = {col: preprocess(dataset_clean[col].values.astype(float))
                for col in dataset_clean.columns}

dataset_prep = pd.DataFrame(preprocessed, index=dataset_clean.index)
dataset_prep.index.name = 'Raman Shift'

print('Preprocessing complete.')
print(f'Shape  : {dataset_prep.shape}')
print(f'Any NaN: {dataset_prep.isna().any().any()}')
print()
norms = np.linalg.norm(dataset_prep.values, axis=0)
print('L2 norms (should all be 1.0):')
for col, n in zip(dataset_prep.columns, norms):
    print(f'  {col:<70} {n:.6f}')

## 6. Functional Data Representation (B-spline, order 4)

Each preprocessed spectrum is projected onto a cubic B-spline basis (order n = 4, degree k = 3) using a least-squares spline fit, with interior knots placed at every 10th wavenumber point — identical settings to `raman_consensus.ipynb`, `raman_adulterated_dataset.ipynb`, `Final/raman_evaluation_dataset.ipynb`, and `Final/raman_intensity_matrix_functional.ipynb`. This converts the discrete, preprocessed intensity vectors into continuous functional objects over the Raman shift domain.

In [ ]:
wn_arr = dataset_prep.index.values.astype(float)

n = 4          # spline order (order 4 = cubic)
k = n - 1      # spline degree = 3

# Interior knots: every 10th point, excluding the boundary
interior_knots = wn_arr[10:-10:10]
t = np.concatenate(([wn_arr[0]] * (k + 1), interior_knots, [wn_arr[-1]] * (k + 1)))

functional_coeffs = {}   # spline coefficients per spectrum
functional_curves = {}   # spline-evaluated curve per spectrum, on the original grid

for col in dataset_prep.columns:
    y = dataset_prep[col].values
    spl = make_lsq_spline(wn_arr, y, t=t, k=k)
    functional_coeffs[col] = spl.c
    functional_curves[col] = spl(wn_arr)

dataset_functional = pd.DataFrame(functional_curves, index=dataset_prep.index)
dataset_functional.index.name = 'Raman Shift'

print(f'B-spline basis: order n = {n} (degree k = {k})')
print(f'Interior knots : {len(interior_knots)}  ->  total basis functions = {len(t) - k - 1}')
print(f'Functional data shape: {dataset_functional.shape}')

max_dev = np.max(np.abs(dataset_functional.values - dataset_prep.values), axis=0)
print(f'Max abs deviation (spline vs discrete preprocessed): mean={max_dev.mean():.2e}, max={max_dev.max():.2e}')

In [ ]:
examples = list(dataset_prep.columns[:3])
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, examples):
    ax.plot(wn_arr, dataset_prep[col].values, alpha=0.5, linewidth=0.8, color='steelblue', label='Preprocessed')
    ax.plot(wn_arr, dataset_functional[col].values, linewidth=1.2, color='crimson', label='B-spline fit')
    ax.set_title(col[:40], fontsize=8)
    ax.set_xlabel('Raman Shift (cm⁻¹)')
axes[0].set_ylabel('Normalised Intensity (a.u.)')
axes[0].legend(fontsize=8)
plt.suptitle('Functional Representation: B-spline Fit vs Preprocessed Data', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Intensity Statistics (Functional, Preprocessed)

In [ ]:
meta_indexed = meta_df.set_index('Column')

stats = pd.DataFrame({
    'Brand':          [meta_indexed.loc[c, 'Brand'] for c in dataset_functional.columns],
    'Milk Type':      [meta_indexed.loc[c, 'Milk Type'] for c in dataset_functional.columns],
    'Condition':      [meta_indexed.loc[c, 'Condition'] for c in dataset_functional.columns],
    'Min':            dataset_functional.min().values,
    'Max':            dataset_functional.max().values,
    'Mean':           dataset_functional.mean().values,
    'Std':            dataset_functional.std().values,
    'SNR (mean/std)': (dataset_functional.mean() / dataset_functional.std()).values,
}, index=dataset_functional.columns)

print(stats.round(5).to_string())

## 8. Overview Plot — All Processed Spectra

In [ ]:
BRAND_COLOR = {'Amul': '#1f77b4', 'Heritage': '#ff7f0e', 'Nandani': '#2ca02c', 'Nestle': '#d62728'}

fig, ax = plt.subplots(figsize=(14, 5))
for col in dataset_functional.columns:
    brand = meta_indexed.loc[col, 'Brand']
    ax.plot(wn_arr, dataset_functional[col], color=BRAND_COLOR.get(brand, 'grey'),
            linewidth=0.8, alpha=0.6)

from matplotlib.patches import Patch
handles = [Patch(color=c, label=b) for b, c in BRAND_COLOR.items()]
ax.legend(handles=handles, fontsize=9, title='Brand', loc='upper right')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title(f'All {dataset_functional.shape[1]} Valid Samples — Preprocessed & Functional (SG + ALS + L² norm + B-spline)')
plt.tight_layout()
plt.show()

## 9. Save Valid Samples Combined Dataset to Excel

In [ ]:
out_path = 'raman_valid_samples_combined.xlsx'

processing_report = pd.DataFrame({
    'Step': ['Loaded (all columns)', 'Non-standard laser removed', 'Duplicates removed', 'Final valid samples'],
    'Count': [len(spectrum_cols), len(exclude), len(to_drop), dataset_clean.shape[1]],
})

with pd.ExcelWriter(out_path) as writer:
    dataset_clean.to_excel(writer, sheet_name='Spectra Raw')
    dataset_prep.to_excel(writer, sheet_name='Spectra Preprocessed')
    dataset_functional.to_excel(writer, sheet_name='Spectra Functional')
    meta_df.to_excel(writer, sheet_name='Metadata', index=False)
    stats.to_excel(writer, sheet_name='Statistics')
    processing_report.to_excel(writer, sheet_name='Processing Report', index=False)

print(f'Saved → {out_path}')
print(f'  Sheet "Spectra Raw"          : {dataset_clean.shape[1]} spectra × {dataset_clean.shape[0]} wn points (raw counts, 532 nm only, deduplicated)')
print(f'  Sheet "Spectra Preprocessed" : {dataset_prep.shape[1]} spectra × {dataset_prep.shape[0]} wn points (SG + ALS + L² norm)')
print(f'  Sheet "Spectra Functional"   : {dataset_functional.shape[1]} spectra × {dataset_functional.shape[0]} wn points (B-spline order 4)')
print(f'  Sheet "Metadata"             : {len(meta_df)} rows')
print(f'  Sheet "Statistics"           : {len(stats)} rows')
print(f'  Sheet "Processing Report"    : {len(processing_report)} rows')